# Gutendex ETL Project

This project gets book data from the first five pages of the Gutendex API.

The required fields are selected and saved in a SQLite database.

At the end, several SQL reports are created from the stored data.

In [1]:
import requests
import pandas as pd
import sqlite3

In [2]:
# url = "https://gutendex.com/books/?page=1"

# response = requests.get(url)

# data = response.json()

# print("keys:", data.keys())

# print("Number of books:", len(data["results"]))

# first_book = data["results"][0]
# print(first_book.keys())

## 1. Extract Data

Book data is fetched from pages 1 to 5 of the Gutendex API.

In [3]:
all_books = []

for page in range(1, 6):
    url = f"https://gutendex.com/books/?page={page}"

    response = requests.get(url)
    response.raise_for_status()

    page_data = response.json()
    books = page_data["results"]

    all_books.extend(books)

    print(f"Page {page}: {len(books)} books")

print("Total books:", len(all_books))

Page 1: 32 books
Page 2: 32 books
Page 3: 32 books
Page 4: 32 books
Page 5: 32 books
Total books: 160


## 2. Transform Data

The required book fields and categories are selected from the data.

In [4]:
books_data = []
categories_data = []

for book in all_books:
    books_data.append({
        "id": book["id"],
        "title": book["title"],
        "media_type": book["media_type"],
        "download_count": book["download_count"]
    })

    for bookshelf in book["bookshelves"]:
        if bookshelf.startswith("Category:"):
            category_name = bookshelf.replace(
                "Category:",
                "",
                1
            ).strip()

            categories_data.append({
                "book_id": book["id"],
                "category": category_name
            })

print("Book records:", len(books_data))
print("Category records:", len(categories_data))

Book records: 160
Category records: 552


### Create DataFrames

The transformed records are converted to Pandas DataFrames.

In [5]:
books_df = pd.DataFrame(books_data)
categories_df = pd.DataFrame(categories_data)

print("Books shape:", books_df.shape)
print("Categories shape:", categories_df.shape)

display(books_df.head())
display(categories_df.head())

Books shape: (160, 4)
Categories shape: (552, 2)


,id,title,media_type,download_count
0,2701,"Moby Dick; Or, The Whale",Text,160099
1,1342,Pride and Prejudice,Text,136926
2,1513,Romeo and Juliet,Text,103120
3,2641,A Room with a View,Text,101658
4,2554,Crime and Punishment,Text,93752


,book_id,category
0,2701,Adventure
1,2701,American Literature
2,2701,Classics of Literature
3,2701,Novels
4,1342,British Literature


### Checking duplicates, missing values, data types, and spaces



In [6]:
print("Number of same ID:", books_df["id"].duplicated().sum())
print("Number of same Category:", categories_df.duplicated().sum())

Number of same ID: 0
Number of same Category: 0


In [7]:
print("Missing values in books:")
print(books_df.isnull().sum())
print("******")
print("Missing values in categories:")
print(categories_df.isnull().sum())

Missing values in books:
id                0
title             0
media_type        0
download_count    0
dtype: int64
******
Missing values in categories:
book_id     0
category    0
dtype: int64


In [8]:
print(books_df.info())
print(categories_df.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 160 entries, 0 to 159
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id              160 non-null    int64 
 1   title           160 non-null    object
 2   media_type      160 non-null    object
 3   download_count  160 non-null    int64 
dtypes: int64(2), object(2)
memory usage: 5.1+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 552 entries, 0 to 551
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   book_id   552 non-null    int64 
 1   category  552 non-null    object
dtypes: int64(1), object(1)
memory usage: 8.8+ KB
None


In [9]:
print("extra spaces checking:",
      (books_df["title"] != books_df["title"].str.strip()).sum(),
      (books_df["media_type"] != books_df["media_type"].str.strip()).sum(),
      (categories_df["category"] != categories_df["category"].str.strip()).sum())

extra spaces checking: 0 0 0


## 3. Load Data into SQLite

#### with two tables: books and category.

In [10]:
conn = sqlite3.connect("data/database/gutendex.db")
cursor = conn.cursor()

cursor.execute("PRAGMA foreign_keys = ON")

cursor.execute("DROP TABLE IF EXISTS category")
cursor.execute("DROP TABLE IF EXISTS books")

cursor.execute("""
CREATE TABLE books (
    id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    media_type TEXT,
    download_count INTEGER
)
""")

cursor.execute("""
CREATE TABLE category (
    book_id INTEGER NOT NULL,
    category TEXT NOT NULL,
    PRIMARY KEY (book_id, category),
    FOREIGN KEY (book_id) REFERENCES books(id)
)
""")

conn.commit()

print("Database and tables created.")

Database and tables created.


In [11]:
# print(books_df.columns)
# print(categories_df.columns)
categories_df.shape

(552, 2)

### Insert Data

In [12]:
books_rows = list(books_df.itertuples(index=False, name=None))
category_rows = list(categories_df.itertuples(index=False, name=None))

cursor.executemany("""
INSERT INTO books (id, title, media_type, download_count)
VALUES (?, ?, ?, ?)
""", books_rows)

cursor.executemany("""
INSERT INTO category (book_id, category)
VALUES (?, ?)
""", category_rows)

conn.commit()

print("Data inserted")

Data inserted


### Check Data



In [13]:
print(cursor.execute("SELECT COUNT(*) FROM books").fetchone())

print(cursor.execute("SELECT COUNT(*) FROM category").fetchone())



(160,)
(552,)


## 4. SQL Reports

### 1. Most Downloaded Books

In [14]:
pd.read_sql_query("""
SELECT id, title, media_type, download_count
FROM books
ORDER BY download_count DESC
LIMIT 10;
""", conn)

,id,title,media_type,download_count
0,2701,"Moby Dick; Or, The Whale",Text,160099
1,1342,Pride and Prejudice,Text,136926
2,1513,Romeo and Juliet,Text,103120
3,2641,A Room with a View,Text,101658
4,2554,Crime and Punishment,Text,93752
5,11,Alice's Adventures in Wonderland,Text,82955
6,1184,The Count of Monte Cristo,Text,77916
7,34413,The Love Letters of Mary Wollstonecraft to Gil...,Text,77483
8,2465,Carmen,Text,76803
9,6133,"The Extraordinary Adventures of Arsène Lupin, ...",Text,75744


### 2. Number of Books per Category

In [15]:
pd.read_sql_query("""
SELECT category, COUNT(*) AS book_count
FROM category
GROUP BY category
ORDER BY book_count DESC;
""", conn)

,category,book_count
0,Novels,93
1,British Literature,74
2,Romance,73
3,Classics of Literature,49
4,"Crime, Thrillers and Mystery",41
5,Adventure,34
6,Biographies,27
7,Historical Novels,20
8,American Literature,20
9,French Literature,14


### 3. Average Downloads per Category


In [16]:
pd.read_sql_query("""
SELECT
    c.category,
    ROUND(AVG(b.download_count), 2) AS average_download_count
FROM category c
JOIN books b
    ON c.book_id = b.id
GROUP BY c.category
ORDER BY average_download_count DESC;
""", conn)

,category,average_download_count
0,Russian Literature,70114.00
1,Music,68850.00
2,Plays/Films/Dramas,67931.25
3,Classics of Literature,55689.18
4,Children & Young Adult Reading,51335.20
5,British Literature,50169.26
6,Novels,49617.30
7,"Crime, Thrillers and Mystery",49480.54
8,American Literature,49309.95
9,French Literature,49136.86


### 4. Books with More Than Three Categories


In [17]:
pd.read_sql_query("""
SELECT
    b.id,
    b.title,
    COUNT(c.category) AS category_count
FROM books b
JOIN category c
    ON b.id = c.book_id
GROUP BY b.id, b.title
HAVING COUNT(c.category) > 3
ORDER BY category_count DESC;
""", conn)

,id,title,category_count
0,831,Four Arthurian Romances,6
1,2160,The Expedition of Humphry Clinker,6
2,345,Dracula,5
3,589,Catriona,5
4,1695,The Man Who Was Thursday: A Nightmare,5
...,...,...,...
75,52404,The Girl Philippa,4
76,56156,Venus im Pelz,4
77,59828,"The String of Pearls; Or, The Barber of Fleet ...",4
78,62215,Le Fantôme de l'Opéra,4


### 5. Most Downloaded Book per Category

In [18]:
pd.read_sql_query("""
SELECT
    c.category,
    b.id,
    b.title,
    b.download_count
FROM category c
JOIN books b
    ON c.book_id = b.id
WHERE b.download_count = (
    SELECT MAX(b2.download_count)
    FROM category c2
    JOIN books b2
        ON c2.book_id = b2.id
    WHERE c2.category = c.category
)
ORDER BY c.category;
""", conn)

,category,id,title,download_count
0,Adventure,2701,"Moby Dick; Or, The Whale",160099
1,American Literature,2701,"Moby Dick; Or, The Whale",160099
2,Biographies,34413,The Love Letters of Mary Wollstonecraft to Gil...,77483
3,British Literature,1342,Pride and Prejudice,136926
4,Children & Young Adult Reading,11,Alice's Adventures in Wonderland,82955
5,Classics of Literature,2701,"Moby Dick; Or, The Whale",160099
6,"Crime, Thrillers and Mystery",2554,Crime and Punishment,93752
7,Encyclopedias/Dictionaries/Reference,27509,The 2006 CIA World Factbook,43777
8,"Essays, Letters & Speeches",34413,The Love Letters of Mary Wollstonecraft to Gil...,77483
9,French Literature,1184,The Count of Monte Cristo,77916


## Finish


In [19]:
conn.close()
print("Database connection closed.")

Database connection closed.
